# 1. 匯入套件與讀取資料

In [667]:
from sklearn import set_config

set_config(display="text")

In [668]:
import pandas as pd

# 讀取上一堂使用的電商 Session 資料
df = pd.read_csv("baseline_ml_data.csv")

# 查看資料大小
print("資料大小：", df.shape)

# 查看前 5 筆資料
df.head()

資料大小： (3012, 13)


,session_id,age,monthly_income,device,traffic_source,campaign,segment,session_hour,is_weekend,page_view,add_to_cart,purchase_amount,purchased
0,S00001,34.0,48887.0,NaN,ads,search,growth,10,0,5,1,0.0,0
1,S00002,36.0,77288.0,mobile,ads,search,new,19,0,7,0,0.0,0
2,S00003,22.0,38869.0,mobile,ads,brand,new,10,1,4,0,0.0,0
3,S00004,28.0,38820.0,mobile,ads,summer_sale,vip,15,0,5,0,0.0,0
4,S00005,30.0,76136.0,mobile,email,NaN,new,16,0,3,0,0.0,0


In [669]:
# 移除完全重複的資料
df = df.drop_duplicates().copy()

# 數值欄位使用中位數補缺失值
df["age"] = df["age"].fillna(df["age"].median())
df["monthly_income"] = df["monthly_income"].fillna(
    df["monthly_income"].median()
)

# 類別欄位缺失時補成「未知」
df["device"] = df["device"].fillna("未知")
df["traffic_source"] = df["traffic_source"].fillna("未知")
df["campaign"] = df["campaign"].fillna("未知")
df["segment"] = df["segment"].fillna("未知")

# 確認缺失值狀況
print(df.isna().sum())

session_id         0
age                0
monthly_income     0
device             0
traffic_source     0
campaign           0
segment            0
session_hour       0
is_weekend         0
page_view          0
add_to_cart        0
purchase_amount    0
purchased          0
dtype: int64


In [670]:
# 數值型 Feature
numeric_columns = [
    "age",
    "monthly_income",
    "session_hour",
    "is_weekend",
    "page_view",
    "add_to_cart"
]

# 類別型 Feature
categorical_columns = [
    "device",
    "traffic_source",
    "campaign",
    "segment"
]

# 原始模型 Feature
raw_feature_columns = numeric_columns + categorical_columns

print("數值欄位：", numeric_columns)
print("類別欄位：", categorical_columns)

數值欄位： ['age', 'monthly_income', 'session_hour', 'is_weekend', 'page_view', 'add_to_cart']
類別欄位： ['device', 'traffic_source', 'campaign', 'segment']


In [671]:
# 先只取模型真正要使用的 Feature
feature_df = df[raw_feature_columns].copy()

# 將類別文字欄位轉成 One-Hot 數字欄位
encoded_features = pd.get_dummies(
    feature_df,
    columns=categorical_columns,
    dtype=int
)

# 記住 One-Hot 後真正的模型欄位
feature_columns = encoded_features.columns.tolist()

# 查看轉換後資料
print("One-Hot 後 Feature 數量：", len(feature_columns))
encoded_features.head()

One-Hot 後 Feature 數量： 25


,age,monthly_income,session_hour,is_weekend,page_view,add_to_cart,device_desktop,device_mobile,device_tablet,device_未知,...,campaign_brand,campaign_none,campaign_search,campaign_summer_sale,campaign_vip_offer,campaign_未知,segment_growth,segment_new,segment_vip,segment_未知
0,34.0,48887.0,10,0,5,1,0,0,0,1,...,0,0,1,0,0,0,1,0,0,0
1,36.0,77288.0,19,0,7,0,0,1,0,0,...,0,0,1,0,0,0,0,1,0,0
2,22.0,38869.0,10,1,4,0,0,1,0,0,...,1,0,0,0,0,0,0,1,0,0
3,28.0,38820.0,15,0,5,0,0,1,0,0,...,0,0,0,1,0,0,0,0,1,0
4,30.0,76136.0,16,0,3,0,0,1,0,0,...,0,0,0,0,0,1,0,1,0,0


## Pipeline

In [672]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
purchased_mask = df["purchased"] == 1
#使用前面已經 One-Hot 完成的資料
X_regression = encoded_features.loc[purchased_mask]
#y是模型要預測的購買金額
y_regression = df.loc[purchased_mask, "purchase_amount"]
regression_pipeline = Pipeline([("scaler",StandardScaler()),
                                ("regression",LinearRegression())])

regression_pipeline

Pipeline(steps=[('scaler', StandardScaler()),
                ('regression', LinearRegression())])

In [673]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
X_train,X_test,y_train,y_test = train_test_split(
    X_regression,
    y_regression,
    test_size=0.2)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((330, 25), (83, 25), (330,), (83,))

In [674]:
regression_pipeline.fit(X_train, y_train)
y_pred = regression_pipeline.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
print("測試集 MSE:", mse)

測試集 MSE: 53786.81279731562


## Pipeline 套用 Logistic Regression

In [675]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

In [676]:
X_classfication = encoded_features
y_classfication = df["purchased"]
classification_model = Pipeline([("scaler", StandardScaler()), 
                                 ("classifier", LogisticRegression())])
classification_model

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier', LogisticRegression())])

In [677]:
X_train,X_test,y_train,y_test = train_test_split(
    X_classfication,y_classfication,
    test_size=0.2,

    stratify=y_classfication)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((2400, 25), (600, 25), (2400,), (600,))

In [678]:
classification_model.fit(X_train, y_train)
y_pred = classification_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred) #越高越好 1 = 100% 越接近1越好
print(f"Accuracy: {accuracy:.2%}")

Accuracy: 86.00%


In [683]:
from sklearn.model_selection import cross_val_score
regression_scores = cross_val_score(
    regression_pipeline,
    X_regression,
    y_regression,
    cv=5,
    scoring="neg_mean_squared_error"
)
regression_mse_scores = -regression_scores
print("每次 MSE:")
print(regression_mse_scores)
print(f"平均 MSE: {regression_mse_scores.mean():.2f}")


每次 MSE:
[69688.56949361 40960.0328191  57294.77691209 59697.8241217
 50289.49135158]
平均 MSE: 55586.14


In [686]:
from sklearn.model_selection import cross_val_score
regression_scores = cross_val_score(
    regression_pipeline,
    X_regression,
    y_regression,
    cv=5,
    scoring="neg_mean_squared_error"
)
regression_mse_scores = -regression_scores
print("每次 MSE:") 
print(regression_mse_scores)
print(f"平均 MSE:{regression_mse_scores.mean() :.2f}")

每次 MSE:
[69688.56949361 40960.0328191  57294.77691209 59697.8241217
 50289.49135158]
平均 MSE:55586.14


## 分類模型做 Cross-Validation

In [696]:
# 使用 5-Fold Cross-Validation 評估分類 Pipeline
classification_scores = cross_val_score(
    classification_model,
    X_classfication,
    y_classfication,
    cv=5,
    scoring="accuracy"
)

print("每次 Accuracy：")
print(classification_scores)

print(
    f"\n平均 Accuracy："
    f"{classification_scores.mean():.2%}"
)

每次 Accuracy：
[0.85166667 0.865      0.85333333 0.86166667 0.86      ]

平均 Accuracy：85.83%


In [697]:
import joblib
classification_model.fit(X_classfication, y_classfication)
model_package = {
    "model": classification_model,
    "feature_columns": X_classfication.columns.tolist()
}
joblib.dump(model_package, "model_package.joblib")

['model_package.joblib']